# SECCIÓN 1
## ¿Qué es un dataset desbalanceado y por qué es un problema?
Un dataset desbalanceado es un conjunto de datos, en el que el número de observaciónes es dispar en la clase de los datos que tenemos para distinguir. En estos conjuntos de datos, la disparidad es tal que unas clases de datos serán mucho más frecuentes que otras. Por lo tanto esto produce que las predicciónes que hará nuestro modelo favorecerán más a la clase mayoritaria, obteniendo malas predicciónes sobre la clase minoritaria 

## ¿Qué métrica es más importante para detectar fraude: precisión, recall o accuracy? ¿Por qué?

La más importante es el recall pues si hacemos las comparación entre los tres, podemos explicarlo de la siguiente manera:

El recall mide cuantos fraudes de los existentes se lograron detectar. La precision indica cauntos fraudes realmente lo son de los que se ha marcado como fraude. Y el accuracy indica el porcentaje de cuantas decisiones se acertó en clasificar como fraude o no fraude. 

El recall se vuelve entonces más importante pues aunque en la clasificación de fraudes se hallan marcado algunas falsas alarmas, la posibilidad de reconocer todos los fraudes es mayor. Caso contrario a las otras dos métricas, pues en la precision, aunque todos los fraudes que se detectaron realmente sean fraude, no se establece si se dejó pasar algún fraude como no fraude. En el caso de accuracy se pueden generar problemas, pues si en general hay pocos fraudes, entonces se tendrá una métrica alta, aunque se hayan dejado pasar la mayor parte de los fraudes.

## Menciona 3 features útiles para detectar posibles cuentas mula

Transacciónes rápidas donde el dinero sale igual que como entra, entre múltiples cuentas sin un propósito comercial definido.

Cuentas con un gran volumen de transacciónes con importes por debajo del límite de reporte 

Cuentas personales que actúan como cuenta empresarial, recibiendo fondos de múltiples cuentas no relacionadas y enviando estos a unos cuantos destinatarias posteriormente.

## Explica qué es overfitting

Es cuando un modelo de ML es entrenado demasiadas veces con los mismos datos, lo cual ocasionas que el modelo aprenda peculiaridades como ruido o patrones específicos del grupo de dato usado para el entrenamiento. Ocasionado que la predición sea mala para un grupo de datos nuevos debido al sobreajuste que aprendió el modelo por el uso continuo de los mismos datos durante el entrenamiento. 

# Sección 2

In [34]:
import pandas as pd
from io import StringIO
import numpy as np

In [35]:
datos = """account_id,age,transactions_last_24h,total_received_last_7d,is_mule
1,22,25,15000,1
2,45,3,800,0
3,19,40,22000,1
4,35,5,600,0
5,28,12,4000,0
6,21,30,18000,1
7,50,2,200,0
8,23,18,9000,0
9,20,33,20000,1
10,31,8,1500,0"""

df = pd.read_csv(StringIO(datos))

# Guardar como CSV
df.to_csv('datos.csv', index=False)

## Carga los datos en Python y realiza limpieza básica.

In [36]:
df = pd.read_csv('datos.csv')

print(f"Dimension: {df.shape}")
print(df.head(10)) # imprimir las primeras n filas

Dimension: (10, 5)
   account_id  age  transactions_last_24h  total_received_last_7d  is_mule
0           1   22                     25                   15000        1
1           2   45                      3                     800        0
2           3   19                     40                   22000        1
3           4   35                      5                     600        0
4           5   28                     12                    4000        0
5           6   21                     30                   18000        1
6           7   50                      2                     200        0
7           8   23                     18                    9000        0
8           9   20                     33                   20000        1
9          10   31                      8                    1500        0


In [37]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   account_id              10 non-null     int64
 1   age                     10 non-null     int64
 2   transactions_last_24h   10 non-null     int64
 3   total_received_last_7d  10 non-null     int64
 4   is_mule                 10 non-null     int64
dtypes: int64(5)
memory usage: 528.0 bytes
None


In [38]:
print(df.isnull().sum()) #hallar valores nulos en cada columna

account_id                0
age                       0
transactions_last_24h     0
total_received_last_7d    0
is_mule                   0
dtype: int64


In [39]:
# Creación de rangos de valores esperados por columna
print("Edades fuera de rango [18, 100]:") #rango de edades esperadas
print(df[(df['age'] < 18) | (df['age'] > 100)])

print("\nMontos recibidos negativos:") #verificar que no haya transferencias negativas
print(df[df['total_received_last_7d'] < 0])

Edades fuera de rango [18, 100]:
Empty DataFrame
Columns: [account_id, age, transactions_last_24h, total_received_last_7d, is_mule]
Index: []

Montos recibidos negativos:
Empty DataFrame
Columns: [account_id, age, transactions_last_24h, total_received_last_7d, is_mule]
Index: []


In [40]:
# Verificar y eliminar filas duplicadas
duplicados = df.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

if duplicados > 0:
    df = df.drop_duplicates()

# Verificar valores nulos (casillas sin datos)
print(f"Valores nulos después de limpieza:")
print(df.isnull().sum())

# Corregir datos en texto a número ("1", "2", "3" a 1, 2, 3)
df['account_id'] = df['account_id'].astype(int)
df['is_mule'] = df['is_mule'].astype(int)

Registros duplicados: 0
Valores nulos después de limpieza:
account_id                0
age                       0
transactions_last_24h     0
total_received_last_7d    0
is_mule                   0
dtype: int64


In [41]:
df.to_csv('datos_limpios.csv', index=False)

## Crea dos nuevas features basadas en los datos existentes.

In [42]:
df = pd.read_csv('datos_limpios.csv')

In [43]:
#Feature para relacionar transacciónes con valor y cantidad de transacción, por posible sospecha de cuentas mula

def patron_transacciones_sospechosas(fila):
    transacciones = fila['transactions_last_24h']
    monto_total = fila['total_received_last_7d']
    monto_por_transaccion = monto_total / transacciones if transacciones > 0 else 0
    
    # PATRÓN MULA: Muchas transacciones (más de 15) y montos altos (más de 10,000)
    if transacciones >= 15 and monto_total >= 10000:
        return "MULA"
    
    # PATRÓN SOSPECHOSO: Transacciones moderadas pero montos muy altos
    elif transacciones >= 10 and monto_total >= 15000:
        return "SOSPECHOSO"
    
    # PATRÓN NORMAL: Pocas transacciones y montos bajos
    else:
        return "NORMAL"

df['transaction_pattern'] = df.apply(patron_transacciones_sospechosas, axis=1)

In [44]:
#feature para definir que tan sospechosa puede ser una cuenta en base a edad y cantidad de dinero recibida
def patron_sospechoso(fila):
    edad = fila['age']
    monto = fila['total_received_last_7d']
    
    # Joven (<=25) con monto alto (>10,000) = muy sospechoso
    if edad <= 25 and monto > 10000:
        return 2  # Muy sospechoso
    # Adulto joven (26-35) con monto muy alto (>15,000) = sospechoso
    elif edad <= 35 and monto > 15000:
        return 1  # Sospechoso
    else:
        return 0  # Normal

df['suspicious_pattern'] = df.apply(patron_sospechoso, axis=1)

In [45]:
df.to_csv('datos_con_nuevas_features.csv', index=False)

## Entrena un modelo de clasificación (Logistic Regression, Random Forest o Decision Tree)

### Modelo por bósque aleatorio 
Se utilizó este modelo, pues al usar un arbol de decision simple, el modelo solo daba relevancia a un feature para hacer las predicciones.

In [46]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib

df = pd.read_csv('datos_con_nuevas_features.csv')

#Uso de features originales, más las dos creadas
features = ['age', 'transactions_last_24h', 'total_received_last_7d', 
           'suspicious_pattern', 'transaction_pattern']
X = df[features]
y = df['is_mule']

le = LabelEncoder()
X_encoded = X.copy()
X_encoded['transaction_pattern'] = le.fit_transform(X['transaction_pattern']) #transformar valores de MULA, SOSPECHOS, NORMAL a número.
#En el data set no hay SOSPECHOSO por lo que solo se tendrá MULA=0 y NORMAL=1

#Se dividen los datos para tener una parte del dataset como entrenamiento y los restantes como predicción para probar el modelo
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

#Configurar el modelo de arbol aleatorio
rf_model = RandomForestClassifier(
    n_estimators=100,      # 100 árboles (asesores)
    max_depth=5,           # Cada árbol tiene máximo 5 niveles de profundidad
    min_samples_split=5,   # Se necesitan 5 ejemplos para crear una regla
    min_samples_leaf=2,    # Cada hoja debe tener al menos 2 ejemplos
    random_state=42,       # Para que los resultados sean reproducibles
    class_weight='balanced' #Presta más atención a las mulas (clase minoritaria), evitando el overfiting
)
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,5
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [47]:
#Visualizador de importancia en cada feature para llegar a la clasificación final 
importancia = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importancia
}).sort_values('Importance', ascending=False)

print(feature_importance_df)

                  Feature  Importance
0                     age    0.296296
1   transactions_last_24h    0.209877
2  total_received_last_7d    0.185185
4     transaction_pattern    0.160494
3      suspicious_pattern    0.148148


In [48]:
#Predicción del modelo
df['prediccion_rf'] = rf_model.predict(X_encoded)
df['probabilidad_mula_rf'] = rf_model.predict_proba(X_encoded)[:, 1]

resultados = df[['account_id', 'is_mule', 'prediccion_rf', 'probabilidad_mula_rf'] + features]
print(resultados.sort_values('probabilidad_mula_rf', ascending=False))

   account_id  is_mule  prediccion_rf  probabilidad_mula_rf  age  \
5           6        1              1              0.846730   21   
2           3        1              1              0.846730   19   
8           9        1              1              0.846730   20   
0           1        1              1              0.816730   22   
7           8        0              0              0.256730   23   
4           5        0              0              0.101253   28   
1           2        0              0              0.093561   45   
3           4        0              0              0.093561   35   
6           7        0              0              0.093561   50   
9          10        0              0              0.093561   31   

   transactions_last_24h  total_received_last_7d  suspicious_pattern  \
5                     30                   18000                   2   
2                     40                   22000                   2   
8                     33           

## Calcula y muestra estas métricas: Precisión, Recall, Accuracy.

In [49]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

y_pred = rf_model.predict(X_test)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred) 
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")
print(f"Precision: {precision:.2%}") 
print(f"Recall: {recall:.2%}")

print(f"Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 100.00%
Precision: 100.00%
Recall: 100.00%
Matriz de confusión:
[[1 0]
 [0 1]]


## Explica brevemente qué variables parecen más importantes para identificar una cuenta mula

Las variables más importantes por orden, según el modelos serán, edad>transacciones en 24HRS> monto recibido en 7 días.

Esto puede explicarse pues la gente más jóven puede tener una tendencias mayor a caer en esquemas de mula, debido a la dificultad de obtener dinero o trabajo estable, pues se puede tratar de estudiantes o gente que inicia su vida laboral. La cantidad de transacciónes en 24HRS es la segunda más importante, pues las cuentas mulas generalmente procesan una gran cantidad de transacciónes en un tiempo muy corto y el monto recibido en 7 día puede indicar que se está usando una cuenta como mula por la gran cantidad de dinero, sin embargo esta debe ser comparada con las dos anteriores, pues una persona de mayor edad con muy pocas transacciónes y grandes movimientos de dinero descartaría la cuenta como cuenta mula. 


# Sección 3

## Si el modelo genera muchas alertas pero detecta pocas cuentas mula reales, ¿qué ajustarías?
Ya que esto sería un problema de presicion, se ajustaría la linea "class_weight='balanced'", pues está escrita para poner más atención a las mulas, lo cual marcaría más cuentas como mulas, aunque sean normales, pues se diseñó buscando un recall alto. 
Esto debería cambiarse a algo del estilo "class_weight={0: 1, 1: 1}" que haría más balanceado el tratamiento de las cuentas, sin embargo, esto podría generar que haya cuentas mula que no se detecten, pues solo se marcarán las cuentas en las que se tenga mayor certeza de ser cuenta mula. 

## Si tuvieras un dataset con solo 0.2% de cuentas mula, ¿qué técnicas usarías para manejar el desbalance?

Usaría un árbol aleatorio balanceado, pues al entrenar se pueden tomar todas las cuentas mula y comparar con la misma cantidad de cuentas normales tomadas de manera aleatoria, por lo que cada arbol estaría enfocado en detectar las cuentas mula y al emplerar el modelo en predicciónes, los arboles estarían más balanceados en el momento de etiquetar una cuenta como mula o no mula.